# Separate image pipeline — skin disease and chest X-ray datasets

This notebook builds **two independent image-classification pipelines**:

1. Skin-disease images → one classifier using folder names as labels.
2. Chest X-ray images → a separate classifier using folder names as labels.

The image models are intentionally not merged with the tabular disease models. This is an educational baseline, not a diagnostic system. Download the datasets only from sources you are permitted to use and keep them outside Git.


## Expected local folders

After downloading and extracting the datasets, place them in one of these layouts. Each class must be represented by a folder name. The notebook also accepts the original Kaggle folder names shown below.

```text
data/images/
├── skin_disease/<class_name>/*.(jpg|jpeg|png|bmp|webp)
└── xray/<class_name>/*.(jpg|jpeg|png|bmp|webp)
```

Sources listed in the project README:
- [20 Skin Diseases Dataset](https://www.kaggle.com/datasets/haroonalam16/20-skin-diseases-dataset)
- [Skin Disease Dataset 2](https://www.kaggle.com/datasets/pacificrm/skindiseasedataset)
- [Lung Disease X-Ray Dataset](https://www.kaggle.com/datasets/fatemehmehrparvar/lung-disease)


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
IMAGE_ROOT = ROOT / 'data' / 'images'
ARTIFACT_ROOT = ROOT / 'artifacts'
MODEL_ROOT = ARTIFACT_ROOT / 'models'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGE_SIZE = (64, 64)
MAX_IMAGES_PER_CLASS = 200
RANDOM_STATE = 42
print('Project root:', ROOT.resolve())
print('Image root:', IMAGE_ROOT.resolve())


In [ ]:
DATASET_ROOTS = {
    'skin_disease': [IMAGE_ROOT / 'skin_disease', IMAGE_ROOT / 'skin-disease-images'],
    'xray': [IMAGE_ROOT / 'xray', IMAGE_ROOT / 'Lung X-Ray Image'],
}

def resolve_dataset_root(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def collect_image_paths(dataset_root, max_per_class=MAX_IMAGES_PER_CLASS):
    if dataset_root is None:
        return pd.DataFrame(columns=['path', 'label'])
    records = []
    for class_dir in sorted(p for p in dataset_root.iterdir() if p.is_dir()):
        paths = sorted(p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
        records.extend({'path': str(path), 'label': class_dir.name} for path in paths[:max_per_class])
    return pd.DataFrame(records)

manifests = {}
for dataset_name, candidates in DATASET_ROOTS.items():
    dataset_root = resolve_dataset_root(candidates)
    manifests[dataset_name] = collect_image_paths(dataset_root)
    print(f'{dataset_name}: root={dataset_root}, images={len(manifests[dataset_name]):,}, classes={manifests[dataset_name]["label"].nunique()}')
    if not manifests[dataset_name].empty:
        display(manifests[dataset_name]['label'].value_counts().rename('images').to_frame())

if all(manifest.empty for manifest in manifests.values()):
    print('No image folders found yet. Download the datasets and place them under data/images/ before running the training cells.')


In [ ]:
def load_features(manifest):
    features, labels, kept_paths = [], [], []
    for row in manifest.itertuples(index=False):
        try:
            with Image.open(row.path) as image:
                image = ImageOps.fit(image.convert('L'), IMAGE_SIZE)
                features.append(np.asarray(image, dtype=np.float32).reshape(-1) / 255.0)
            labels.append(row.label)
            kept_paths.append(row.path)
        except (OSError, ValueError) as exc:
            warnings.warn(f'Skipping unreadable image {row.path}: {exc}')
    return np.asarray(features, dtype=np.float32), np.asarray(labels), kept_paths

def train_image_pipeline(dataset_name, manifest):
    if manifest.empty:
        return {'dataset': dataset_name, 'status': 'skipped', 'reason': 'image folder not found'}
    X, y, kept_paths = load_features(manifest)
    class_count = len(np.unique(y))
    if class_count < 2:
        return {'dataset': dataset_name, 'status': 'skipped', 'reason': 'at least two class folders are required'}
    if min(pd.Series(y).value_counts()) < 2:
        return {'dataset': dataset_name, 'status': 'skipped', 'reason': 'each class needs at least two readable images'}
    test_size = max(0.2, class_count / len(y))
    X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
        X, y, kept_paths, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )
    pipeline = Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE)),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    metrics = {
        'dataset': dataset_name,
        'status': 'trained',
        'images': int(len(y)),
        'classes': int(class_count),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, predictions)),
        'macro_f1': float(f1_score(y_test, predictions, average='macro')),
    }
    model_path = MODEL_ROOT / f'image_{dataset_name}_logistic_regression.joblib'
    joblib.dump({'pipeline': pipeline, 'image_size': IMAGE_SIZE, 'classes': sorted(np.unique(y).tolist()), 'dataset': dataset_name}, model_path)
    print(f'\n{dataset_name} metrics:', metrics)
    print(classification_report(y_test, predictions, zero_division=0))
    print('Saved:', model_path)
    return metrics

# Each call is a separate model and artifact. No skin image is used in the X-ray model, or vice versa.
results = [train_image_pipeline(name, manifest) for name, manifest in manifests.items()]
results_df = pd.DataFrame(results)
results_df.to_csv(ARTIFACT_ROOT / 'image_pipeline_results.csv', index=False)
results_df


## Interpretation and safety

- The baseline converts each image to a fixed-size grayscale vector and trains a separate balanced logistic-regression model per dataset.
- This is not transfer learning and should not be used for clinical decisions. Image quality, patient-level leakage, class imbalance, and dataset shift require a more rigorous study.
- Review the generated metrics and class-wise report before considering any later model. Keep image models and tabular models as separate validated branches.
